# Prevent Random Disconnect

In [ ]:
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("")
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click()
     }

   btn = document.getElementById('ok')
   if (btn != null){
     console.log("Click reconnect");
     btn.click()
     }
  }

setInterval(ClickConnect,60000)
'''))

print("Done.")

<IPython.core.display.Javascript object>

Done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Install Dependencies

In [ ]:
!python --version

Python 3.12.13


In [ ]:

!pip install torch_geometric thefuzz pandas scikit-learn

# Check Environment

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

PyTorch Version: 2.11.0+cu128
CUDA Available: True
Device Name: Tesla T4


# Data Preparation — V1 (CART + RENT from Anime.csv + User-AnimeReview.csv)

In [ ]:
import pandas as pd
import torch
import numpy as np
from sklearn.model_selection import train_test_split
import os

base_path = '/content/drive/MyDrive/0_DATA_RM'

ANIME_CSV     = os.path.join(base_path, 'Anime.csv')
REVIEW_CSV    = os.path.join(base_path, 'User-AnimeReview.csv')
SYNTHETIC_CSV = os.path.join(base_path, 'synthetic_behavior_logs.csv')

V1_GRAPH_PATH  = os.path.join(base_path, 'mbcgcn_graph_data.pt')
V1_WEIGHT_PATH = os.path.join(base_path, 'mbcgcn_manga_weights.pth')
BL_MF_PATH     = os.path.join(base_path, 'baseline_mf_weights.pth')
BL_GCN_PATH    = os.path.join(base_path, 'baseline_lightgcn_weights.pth')
V2_GRAPH_PATH  = os.path.join(base_path, 'mbcgcn_v2_graph_data.pt')
V2_WEIGHT_PATH = os.path.join(base_path, 'mbcgcn_v2_manga_weights.pth')

print('Config ready.')
for p in [ANIME_CSV, REVIEW_CSV, SYNTHETIC_CSV]:
    print(f'  {p} — {"OK" if os.path.exists(p) else "NOT FOUND"}')

In [ ]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

def to_bipartite_edge_index(user_vals, item_vals, num_users):
    items_off = [i + num_users for i in item_vals]
    return torch.tensor(
        [list(user_vals) + items_off, items_off + list(user_vals)],
        dtype=torch.long)

print('Loading Anime.csv and User-AnimeReview.csv...')
anime_df  = pd.read_csv(ANIME_CSV)
review_df = pd.read_csv(REVIEW_CSV)

manga_df = anime_df[anime_df['source'] == 'Manga'].copy()
rev_df   = review_df[review_df['username'] != '010447'].copy()
rev_df   = rev_df[rev_df['anime_id'].isin(manga_df['anime_id'])]

users = sorted(rev_df['username'].unique())
items = sorted(rev_df['anime_id'].unique())
user_mapping = {u: i for i, u in enumerate(users)}
item_mapping = {m: i for i, m in enumerate(items)}
num_users_v1 = len(user_mapping)
num_items_v1 = len(item_mapping)
print(f'V1 — Users: {num_users_v1}, Items: {num_items_v1}')

cart_df = rev_df[rev_df['my_status'] == 'Plan to Watch']
rent_df = rev_df[rev_df['my_status'].isin(['Watching', 'Completed'])]

def to_pairs(df):
    return list(set(
        (user_mapping[r.username], item_mapping[r.anime_id])
        for r in df.itertuples()
        if r.username in user_mapping and r.anime_id in item_mapping))

cart_pairs = to_pairs(cart_df)
rent_pairs = to_pairs(rent_df)
print(f'CART pairs: {len(cart_pairs)}, RENT pairs: {len(rent_pairs)}')

rent_train, rent_test = train_test_split(rent_pairs, test_size=0.2, random_state=42)

cart_u, cart_i       = zip(*cart_pairs)  if cart_pairs  else ([], [])
rent_tr_u, rent_tr_i = zip(*rent_train)  if rent_train  else ([], [])
rent_te_u, rent_te_i = zip(*rent_test)   if rent_test   else ([], [])

v1_data = {
    'edge_index_cart':       to_bipartite_edge_index(cart_u,    cart_i,    num_users_v1),
    'edge_index_rent_train': to_bipartite_edge_index(rent_tr_u, rent_tr_i, num_users_v1),
    'edge_index_rent_test':  to_bipartite_edge_index(rent_te_u, rent_te_i, num_users_v1),
    'num_users':    num_users_v1,
    'num_items':    num_items_v1,
    'user_mapping': user_mapping,
    'item_mapping': item_mapping,
}
torch.save(v1_data, V1_GRAPH_PATH)
print(f'V1 graph saved -> {V1_GRAPH_PATH}')

In [ ]:
# Data Preparation — V2 (CLICK + ADD_CART + RENT from synthetic_behavior_logs.csv)

In [ ]:
import pandas as pd
import torch

def build_ei_v2(edges, num_users):
    if not edges:
        return torch.empty((2, 0), dtype=torch.long)
    u, m = zip(*edges)
    m_off = [x + num_users for x in m]
    return torch.tensor([list(u) + m_off, m_off + list(u)], dtype=torch.long)

def split_edges(edges, ratio=0.8):
    n = int(len(edges) * ratio)
    return edges[:n], edges[n:]

print('Loading synthetic_behavior_logs.csv...')
df = pd.read_csv(SYNTHETIC_CSV, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

users  = sorted(df['user_id'].unique())
mangas = sorted(df['manga_id'].unique())
user_id_map  = {uid: idx for idx, uid in enumerate(users)}
manga_id_map = {mid: idx for idx, mid in enumerate(mangas)}
n_users = len(user_id_map)
n_items = len(manga_id_map)
print(f'Users: {n_users}, Items: {n_items}')

def extract_v2(event):
    sub = df[df['event_type'] == event].sort_values('timestamp')
    return [(user_id_map[r.user_id], manga_id_map[r.manga_id]) for r in sub.itertuples()]

click_edges = extract_v2('CLICK')
cart_edges  = extract_v2('ADD_CART')
rent_edges  = extract_v2('RENT')
print(f'CLICK: {len(click_edges)}, ADD_CART: {len(cart_edges)}, RENT: {len(rent_edges)}')

c_tr,  c_te  = split_edges(click_edges)
ca_tr, ca_te = split_edges(cart_edges)
r_tr,  r_te  = split_edges(rent_edges)

v2_data = {
    'edge_index_click_train': build_ei_v2(c_tr,  n_users),
    'edge_index_click_test':  build_ei_v2(c_te,  n_users),
    'edge_index_cart_train':  build_ei_v2(ca_tr, n_users),
    'edge_index_cart_test':   build_ei_v2(ca_te, n_users),
    'edge_index_rent_train':  build_ei_v2(r_tr,  n_users),
    'edge_index_rent_test':   build_ei_v2(r_te,  n_users),
    'num_users':   n_users,
    'num_items':   n_items,
    'user_id_map':  user_id_map,
    'manga_id_map': manga_id_map,
}
torch.save(v2_data, V2_GRAPH_PATH)
print(f'Saved V2 graph -> {V2_GRAPH_PATH}')

In [ ]:
# Model Definitions (MB-CGCN V1, V2, Baseline MF, Baseline LightGCN)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import LGConv

class BehaviorGCN(nn.Module):
    def __init__(self, num_layers=1):
        super().__init__()
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])
    def forward(self, x, edge_index):
        embs = [x]
        for conv in self.convs:
            x = conv(x, edge_index)
            embs.append(x)
        return torch.stack(embs, dim=1).mean(dim=1)

# V1: fixed weights 0.1 / 0.9
class MBCGCN_TwoBehaviors(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64, cart_layers=1, rent_layers=1):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        self.gcn_cart = BehaviorGCN(cart_layers)
        self.gcn_rent = BehaviorGCN(rent_layers)
        self.trans_user_cart2rent = nn.Linear(embed_dim, embed_dim, bias=False)
        self.trans_item_cart2rent = nn.Linear(embed_dim, embed_dim, bias=False)
        nn.init.xavier_uniform_(self.trans_user_cart2rent.weight)
        nn.init.xavier_uniform_(self.trans_item_cart2rent.weight)
        self.w_cart = 0.1
        self.w_rent = 0.9
    def forward(self, edge_index_cart, edge_index_rent):
        u0, i0 = self.user_emb.weight, self.item_emb.weight
        nu, ni = u0.size(0), i0.size(0)
        cart_emb = self.gcn_cart(torch.cat([u0, i0]), edge_index_cart)
        cu, ci = torch.split(cart_emb, [nu, ni])
        rent_emb = self.gcn_rent(
            torch.cat([self.trans_user_cart2rent(cu), self.trans_item_cart2rent(ci)]),
            edge_index_rent)
        ru, ri = torch.split(rent_emb, [nu, ni])
        return self.w_cart*cu + self.w_rent*ru, self.w_cart*ci + self.w_rent*ri

# V2: adaptive softmax weights
class MBCGCN_ThreeBehaviors(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64,
                 click_layers=1, cart_layers=1, rent_layers=2, adaptive_weights=True):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.adaptive_weights = adaptive_weights
        self.user_embedding = nn.Embedding(num_users, embed_dim)
        self.item_embedding = nn.Embedding(num_items, embed_dim)
        self.gcn_click = BehaviorGCN(click_layers)
        self.gcn_cart  = BehaviorGCN(cart_layers)
        self.gcn_rent  = BehaviorGCN(rent_layers)
        self.trans_click2cart = nn.Linear(embed_dim, embed_dim, bias=False)
        self.trans_cart2rent  = nn.Linear(embed_dim, embed_dim, bias=False)
        if adaptive_weights:
            self.weight_logits = nn.Parameter(torch.tensor([0.5, 1.5, 3.0]))
        else:
            self.register_buffer('w_click', torch.tensor(0.05))
            self.register_buffer('w_cart',  torch.tensor(0.15))
            self.register_buffer('w_rent',  torch.tensor(0.80))
        nn.init.xavier_uniform_(self.user_embedding.weight)
        nn.init.xavier_uniform_(self.item_embedding.weight)
        nn.init.xavier_uniform_(self.trans_click2cart.weight)
        nn.init.xavier_uniform_(self.trans_cart2rent.weight)
    def get_behavior_weights(self):
        if self.adaptive_weights:
            w = F.softmax(self.weight_logits, dim=0)
            return w[0].item(), w[1].item(), w[2].item()
        return self.w_click.item(), self.w_cart.item(), self.w_rent.item()
    def forward(self, edge_index_click, edge_index_cart, edge_index_rent):
        x = torch.cat([self.user_embedding.weight, self.item_embedding.weight], dim=0)
        x_click = self.gcn_click(x, edge_index_click)
        x_cart  = self.gcn_cart(self.trans_click2cart(x_click), edge_index_cart)
        x_rent  = self.gcn_rent(self.trans_cart2rent(x_cart),   edge_index_rent)
        if self.adaptive_weights:
            w = F.softmax(self.weight_logits, dim=0)
            x_f = w[0]*x_click + w[1]*x_cart + w[2]*x_rent
        else:
            x_f = self.w_click*x_click + self.w_cart*x_cart + self.w_rent*x_rent
        return x_f[:self.num_users], x_f[self.num_users:]

class Baseline_MF(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
    def get_embeddings(self):
        return self.user_emb.weight, self.item_emb.weight

class Baseline_LightGCN(nn.Module):
    def __init__(self, num_users, num_items, embed_dim=64, num_layers=1):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        self.gcn_rent = BehaviorGCN(num_layers)
    def forward(self, edge_index_rent):
        u, i = self.user_emb.weight, self.item_emb.weight
        emb = self.gcn_rent(torch.cat([u, i]), edge_index_rent)
        return torch.split(emb, [u.size(0), i.size(0)])

print('Models defined.')

In [ ]:
# Shared Training Utils (BPR Loss, extract_raw_pairs, evaluate_metrics)

In [ ]:
import torch
import torch.nn.functional as F
import math

def bpr_loss(u_emb, pos_emb, neg_emb, tau=0.05):
    u   = F.normalize(u_emb,   p=2, dim=1)
    pos = F.normalize(pos_emb, p=2, dim=1)
    neg = F.normalize(neg_emb, p=2, dim=1)
    return -torch.mean(F.logsigmoid(
        (torch.sum(u*pos, dim=1) - torch.sum(u*neg, dim=1)) / tau))

def extract_raw_pairs(edge_index_bi, num_users):
    """Bidirectional bipartite [2,2E] -> forward user->item pairs."""
    half = edge_index_bi.size(1) // 2
    fwd  = edge_index_bi[:, :half]
    return fwd[0], fwd[1] - num_users

def evaluate_metrics(u_emb, i_emb, edge_rent_train, edge_rent_test, num_users,
                     ks=(10, 20), num_samples=1000):
    with torch.no_grad():
        tr_u, tr_i = extract_raw_pairs(edge_rent_train, num_users)
        te_u, te_i = extract_raw_pairs(edge_rent_test,  num_users)
        all_te     = torch.unique(te_u)
        n_eval     = min(num_samples, len(all_te))
        eval_users = all_te[torch.randperm(len(all_te))[:n_eval]]

        metrics  = {f'Recall@{k}': 0.0 for k in ks}
        metrics |= {f'NDCG@{k}':   0.0 for k in ks}
        norm_i   = F.normalize(i_emb, p=2, dim=1)
        max_k    = min(max(ks), i_emb.size(0))

        for u in eval_users:
            uid    = u.item()
            scores = torch.matmul(norm_i, F.normalize(u_emb[uid], p=2, dim=0))
            scores[tr_i[tr_u == uid]] = -float('inf')
            true   = set(te_i[te_u == uid].tolist())
            if not true: continue
            top    = torch.topk(scores, max_k).indices.tolist()
            for k in ks:
                hits = [1 if t in true else 0 for t in top[:k]]
                metrics[f'Recall@{k}'] += sum(hits) / len(true)
                dcg  = sum(h/math.log2(i+2) for i,h in enumerate(hits))
                idcg = sum(1/math.log2(i+2) for i in range(min(k, len(true))))
                metrics[f'NDCG@{k}'] += dcg/idcg if idcg else 0

        for key in metrics: metrics[key] /= n_eval
        return metrics

print('Training utils ready.')

In [ ]:
# Train — MB-CGCN V1 (CART + RENT, embed_dim=64, w_cart=0.1, w_rent=0.9)

In [ ]:
import torch, time
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

d = torch.load(V1_GRAPH_PATH, weights_only=False)
num_users = d['num_users']
num_items = d['num_items']
e_cart       = d['edge_index_cart'].to(device)
e_rent       = d['edge_index_rent_train'].to(device)
e_rent_test  = d['edge_index_rent_test'].to(device)

rent_u_raw, rent_i_raw = extract_raw_pairs(e_rent, num_users)
item_counts = torch.bincount(rent_i_raw, minlength=num_items).float()
item_probs  = torch.pow(item_counts + 1.0, 0.75)
item_probs  = (item_probs / item_probs.sum()).to(device)

model_v1  = MBCGCN_TwoBehaviors(num_users, num_items, embed_dim=64,
                                  cart_layers=1, rent_layers=1).to(device)
optimizer = optim.Adam(model_v1.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)

EPOCHS     = 600
BATCH_SIZE = 400000
L2_LAMBDA  = 2e-2
num_inter  = rent_u_raw.size(0)

v1_log = {'loss': [], 'recall10': [], 'ndcg10': [], 'recall20': [], 'ndcg20': []}

for epoch in range(EPOCHS):
    t0 = time.time()
    model_v1.train()
    optimizer.zero_grad()

    u_emb, i_emb = model_v1(e_cart, e_rent)

    n    = min(BATCH_SIZE, num_inter)
    idx  = torch.randperm(num_inter, device=device)[:n]
    u    = rent_u_raw[idx]
    pos  = rent_i_raw[idx]
    neg  = torch.multinomial(item_probs, n, replacement=True)

    loss_bpr = bpr_loss(u_emb[u], i_emb[pos], i_emb[neg])
    loss_reg = 0.5 * (
        model_v1.user_emb.weight[u].norm(2)**2 +
        model_v1.item_emb.weight[pos].norm(2)**2 +
        model_v1.item_emb.weight[neg].norm(2)**2) / n
    loss = loss_bpr + L2_LAMBDA * loss_reg
    loss.backward(); optimizer.step(); scheduler.step()
    torch.cuda.empty_cache()

    if (epoch + 1) % 20 == 0:
        model_v1.eval()
        ue, ie = model_v1(e_cart, e_rent)
        res = evaluate_metrics(ue.cpu(), ie.cpu(),
                               e_rent.cpu(), e_rent_test.cpu(), num_users)
        lr  = scheduler.get_last_lr()[0]
        v1_log['loss'].append(loss.item())
        v1_log['recall10'].append(res['Recall@10'])
        v1_log['ndcg10'].append(res['NDCG@10'])
        v1_log['recall20'].append(res['Recall@20'])
        v1_log['ndcg20'].append(res['NDCG@20'])
        print(f'V1 Epoch {epoch+1:03d}/{EPOCHS} | Loss: {loss.item():.4f} | LR: {lr:.5f} | '
              f'Recall@10: {res["Recall@10"]:.4f} | NDCG@10: {res["NDCG@10"]:.4f} | '
              f'Time: {time.time()-t0:.1f}s')

torch.save(model_v1.state_dict(), V1_WEIGHT_PATH)
print(f'V1 saved -> {V1_WEIGHT_PATH}')

In [ ]:
# Train — Baseline MF + Baseline LightGCN (on V1 RENT data, embed_dim=64)

In [ ]:
import torch, time
import torch.optim as optim

# reuse e_cart, e_rent, e_rent_test, item_probs, num_users, num_items from V1 training cell

mf_log  = {'loss': [], 'recall10': [], 'ndcg10': [], 'recall20': [], 'ndcg20': []}
gcn_log = {'loss': [], 'recall10': [], 'ndcg10': [], 'recall20': [], 'ndcg20': []}

# ── Baseline MF ───────────────────────────────────────────────────────────────
model_mf = Baseline_MF(num_users, num_items, embed_dim=64).to(device)
opt_mf   = optim.Adam(model_mf.parameters(), lr=0.001)
sch_mf   = optim.lr_scheduler.StepLR(opt_mf, step_size=100, gamma=0.5)

for epoch in range(EPOCHS):
    model_mf.train(); opt_mf.zero_grad()
    ue, ie = model_mf.get_embeddings()
    n   = min(BATCH_SIZE, num_inter)
    idx = torch.randperm(num_inter, device=device)[:n]
    u   = rent_u_raw[idx]; pos = rent_i_raw[idx]
    neg = torch.multinomial(item_probs, n, replacement=True)
    loss_bpr = bpr_loss(ue[u], ie[pos], ie[neg])
    loss_reg = 0.5 * (
        model_mf.user_emb.weight[u].norm(2)**2 +
        model_mf.item_emb.weight[pos].norm(2)**2 +
        model_mf.item_emb.weight[neg].norm(2)**2) / n
    loss = loss_bpr + L2_LAMBDA * loss_reg
    loss.backward(); opt_mf.step(); sch_mf.step()
    if (epoch + 1) % 20 == 0:
        model_mf.eval()
        ue, ie = model_mf.get_embeddings()
        res = evaluate_metrics(ue.cpu(), ie.cpu(),
                               e_rent.cpu(), e_rent_test.cpu(), num_users)
        mf_log['loss'].append(loss.item())
        mf_log['recall10'].append(res['Recall@10'])
        mf_log['ndcg10'].append(res['NDCG@10'])
        mf_log['recall20'].append(res['Recall@20'])
        mf_log['ndcg20'].append(res['NDCG@20'])
        print(f'MF  Epoch {epoch+1:03d}/{EPOCHS} | Loss: {loss.item():.4f} | '
              f'Recall@10: {res["Recall@10"]:.4f} | NDCG@10: {res["NDCG@10"]:.4f}')

torch.save(model_mf.state_dict(), BL_MF_PATH)
print(f'Baseline MF saved -> {BL_MF_PATH}')

# ── Baseline LightGCN ─────────────────────────────────────────────────────────
model_gcn = Baseline_LightGCN(num_users, num_items, embed_dim=64, num_layers=1).to(device)
opt_gcn   = optim.Adam(model_gcn.parameters(), lr=0.001)
sch_gcn   = optim.lr_scheduler.StepLR(opt_gcn, step_size=100, gamma=0.5)

for epoch in range(EPOCHS):
    model_gcn.train(); opt_gcn.zero_grad()
    ue, ie = model_gcn(e_rent)
    n   = min(BATCH_SIZE, num_inter)
    idx = torch.randperm(num_inter, device=device)[:n]
    u   = rent_u_raw[idx]; pos = rent_i_raw[idx]
    neg = torch.multinomial(item_probs, n, replacement=True)
    loss_bpr = bpr_loss(ue[u], ie[pos], ie[neg])
    loss_reg = 0.5 * (
        model_gcn.user_emb.weight[u].norm(2)**2 +
        model_gcn.item_emb.weight[pos].norm(2)**2 +
        model_gcn.item_emb.weight[neg].norm(2)**2) / n
    loss = loss_bpr + L2_LAMBDA * loss_reg
    loss.backward(); opt_gcn.step(); sch_gcn.step()
    if (epoch + 1) % 20 == 0:
        model_gcn.eval()
        ue, ie = model_gcn(e_rent)
        res = evaluate_metrics(ue.cpu(), ie.cpu(),
                               e_rent.cpu(), e_rent_test.cpu(), num_users)
        gcn_log['loss'].append(loss.item())
        gcn_log['recall10'].append(res['Recall@10'])
        gcn_log['ndcg10'].append(res['NDCG@10'])
        gcn_log['recall20'].append(res['Recall@20'])
        gcn_log['ndcg20'].append(res['NDCG@20'])
        print(f'GCN Epoch {epoch+1:03d}/{EPOCHS} | Loss: {loss.item():.4f} | '
              f'Recall@10: {res["Recall@10"]:.4f} | NDCG@10: {res["NDCG@10"]:.4f}')

torch.save(model_gcn.state_dict(), BL_GCN_PATH)
print(f'Baseline LightGCN saved -> {BL_GCN_PATH}')

In [ ]:
# Train — MB-CGCN V2 (CLICK + ADD_CART + RENT, embed_dim=64, adaptive weights)

In [ ]:
import torch, time
import torch.optim as optim

d2 = torch.load(V2_GRAPH_PATH, weights_only=False)
n_users2 = d2['num_users']
n_items2 = d2['num_items']
e_click2     = d2['edge_index_click_train'].to(device)
e_cart2      = d2['edge_index_cart_train'].to(device)
e_rent2      = d2['edge_index_rent_train'].to(device)
e_rent2_test = d2['edge_index_rent_test'].to(device)

rent_u2, rent_i2 = extract_raw_pairs(e_rent2, n_users2)
ic2 = torch.bincount(rent_i2, minlength=n_items2).float()
ip2 = torch.pow(ic2 + 1.0, 0.75)
ip2 = (ip2 / ip2.sum()).to(device)

model_v2 = MBCGCN_ThreeBehaviors(
    n_users2, n_items2, embed_dim=64,
    click_layers=1, cart_layers=1, rent_layers=2,
    adaptive_weights=True).to(device)

opt_v2 = optim.Adam(model_v2.parameters(), lr=0.001, weight_decay=2e-2)

EPOCHS_V2  = 600
BATCH_V2   = 400000
num_inter2 = rent_u2.size(0)

v2_log = {'loss': [], 'recall10': [], 'ndcg10': [], 'recall20': [], 'ndcg20': []}

wc, wca, wr = model_v2.get_behavior_weights()
print(f'V2 initial weights: CLICK={wc:.3f}, CART={wca:.3f}, RENT={wr:.3f}')

for epoch in range(EPOCHS_V2):
    t0 = time.time()
    model_v2.train(); opt_v2.zero_grad()
    ue, ie = model_v2(e_click2, e_cart2, e_rent2)
    n   = min(BATCH_V2, num_inter2)
    idx = torch.randperm(num_inter2, device=device)[:n]
    u   = rent_u2[idx]; pos = rent_i2[idx]
    neg = torch.multinomial(ip2, n, replacement=True)
    loss = bpr_loss(ue[u], ie[pos], ie[neg])
    loss.backward(); opt_v2.step()
    torch.cuda.empty_cache()

    if (epoch + 1) % 50 == 0:
        model_v2.eval()
        ue2, ie2 = model_v2(e_click2, e_cart2, e_rent2)
        res = evaluate_metrics(ue2.cpu(), ie2.cpu(),
                               e_rent2.cpu(), e_rent2_test.cpu(), n_users2)
        wc, wca, wr = model_v2.get_behavior_weights()
        v2_log['loss'].append(loss.item())
        v2_log['recall10'].append(res['Recall@10'])
        v2_log['ndcg10'].append(res['NDCG@10'])
        v2_log['recall20'].append(res['Recall@20'])
        v2_log['ndcg20'].append(res['NDCG@20'])
        print(f'V2 Epoch {epoch+1:03d}/{EPOCHS_V2} | Loss: {loss.item():.4f} | '
              f'Recall@10: {res["Recall@10"]:.4f} | NDCG@10: {res["NDCG@10"]:.4f} | '
              f'W=[{wc:.2f},{wca:.2f},{wr:.2f}] | Time: {time.time()-t0:.1f}s')

wc, wca, wr = model_v2.get_behavior_weights()
torch.save({
    'model_state_dict': model_v2.state_dict(),
    'num_users': n_users2, 'num_items': n_items2, 'embed_dim': 64,
    'learned_weights': {'click': wc, 'cart': wca, 'rent': wr},
}, V2_WEIGHT_PATH)
print(f'V2 saved -> {V2_WEIGHT_PATH}')

In [ ]:
# Final evaluation — all 4 models → final_results (used by plots cell below)
print('Running final evaluation...')
final_results = {}

model_mf.eval()
with torch.no_grad():
    ue, ie = model_mf.get_embeddings()
res = evaluate_metrics(ue.cpu(), ie.cpu(), e_rent.cpu(), e_rent_test.cpu(), num_users)
final_results['Baseline MF'] = {k: res[k] for k in ('Recall@10','NDCG@10','Recall@20','NDCG@20')}

model_gcn.eval()
with torch.no_grad():
    ue, ie = model_gcn(e_rent)
res = evaluate_metrics(ue.cpu(), ie.cpu(), e_rent.cpu(), e_rent_test.cpu(), num_users)
final_results['Baseline LightGCN'] = {k: res[k] for k in ('Recall@10','NDCG@10','Recall@20','NDCG@20')}

model_v1.eval()
with torch.no_grad():
    ue, ie = model_v1(e_cart, e_rent)
res = evaluate_metrics(ue.cpu(), ie.cpu(), e_rent.cpu(), e_rent_test.cpu(), num_users)
final_results['MB-CGCN V1'] = {k: res[k] for k in ('Recall@10','NDCG@10','Recall@20','NDCG@20')}

model_v2.eval()
with torch.no_grad():
    ue, ie = model_v2(e_click2, e_cart2, e_rent2)
res = evaluate_metrics(ue.cpu(), ie.cpu(), e_rent2.cpu(), e_rent2_test.cpu(), n_users2)
final_results['MB-CGCN V2'] = {k: res[k] for k in ('Recall@10','NDCG@10','Recall@20','NDCG@20')}

print(f'\n{"Model":20s} | Recall@10 | NDCG@10 | Recall@20 | NDCG@20')
print('-' * 65)
for m, r in final_results.items():
    print(f'{m:20s} | {r["Recall@10"]:.4f}    | {r["NDCG@10"]:.4f}  | {r["Recall@20"]:.4f}    | {r["NDCG@20"]:.4f}')

In [ ]:
from thefuzz import process as fuzz_process

# Load anime name lookup
anime_names_df = pd.read_csv(ANIME_CSV)[['anime_id', 'name']].dropna()
id2name = dict(zip(anime_names_df['anime_id'], anime_names_df['name']))
name2id = {v: k for k, v in id2name.items()}

d_v1 = torch.load(V1_GRAPH_PATH, weights_only=False)
item_map_v1 = d_v1['item_mapping']
rev_item_v1 = {v: k for k, v in item_map_v1.items()}

d_v2 = torch.load(V2_GRAPH_PATH, weights_only=False)
item_map_v2 = d_v2['manga_id_map']
rev_item_v2 = {v: k for k, v in item_map_v2.items()}

all_title_list = list(id2name.values())

def item_recs(title_query, top_k=10):
    match, score = fuzz_process.extractOne(title_query, all_title_list)
    if score < 60:
        print(f'No confident match (best: "{match}", score={score})')
        return
    matched_id = name2id[match]
    print(f'Query matched: "{match}" (score={score})\n')

    with torch.no_grad():
        # MF
        ue_mf, ie_mf = model_mf.get_embeddings()
        # LightGCN
        ue_gcn, ie_gcn = model_gcn(e_rent)
        # V1
        ue_v1, ie_v1 = model_v1(e_cart, e_rent)
        # V2
        ue_v2, ie_v2 = model_v2(e_click2, e_cart2, e_rent2)

    def top_items_v1(ie, matched_id):
        if matched_id not in item_map_v1: return ['(not in V1 graph)']
        idx = item_map_v1[matched_id]
        s = torch.matmul(ie, ie[idx]); s[idx] = -float('inf')
        return [id2name.get(rev_item_v1[i], str(rev_item_v1[i]))
                for i in torch.topk(s, min(top_k, ie.size(0))).indices.tolist()]

    def top_items_v2(ie_v2, matched_id):
        if matched_id not in item_map_v2: return ['(not in V2 graph)']
        idx = item_map_v2[matched_id]
        s = torch.matmul(ie_v2, ie_v2[idx]); s[idx] = -float('inf')
        return [str(rev_item_v2[i])
                for i in torch.topk(s, min(top_k, ie_v2.size(0))).indices.tolist()]

    mf_recs  = top_items_v1(ie_mf,  matched_id)
    gcn_recs = top_items_v1(ie_gcn, matched_id)
    v1_recs  = top_items_v1(ie_v1,  matched_id)
    v2_recs  = top_items_v2(ie_v2,  matched_id)

    w = 32
    print(f'{"#":>3} | {"MF":^{w}} | {"LightGCN":^{w}} | {"MB-CGCN V1":^{w}} | {"MB-CGCN V2":^{w}}')
    print('-' * (w*4 + 16))
    for i in range(top_k):
        def g(lst): return lst[i][:w] if i < len(lst) else ''
        print(f'{i+1:>3} | {g(mf_recs):<{w}} | {g(gcn_recs):<{w}} | {g(v1_recs):<{w}} | {g(v2_recs):<{w}}')
    print()

model_mf.eval(); model_gcn.eval(); model_v1.eval(); model_v2.eval()

print('=== 4-Model Sandbox (item-based) ===')
print('พิมพ์ชื่อมังงะเพื่อดูคำแนะนำจาก 4 โมเดล | พิมพ์ quit เพื่อออก\n')
while True:
    q = input('ชื่อมังงะ: ').strip()
    if q.lower() in ('quit', 'exit', 'q', ''): break
    item_recs(q)

In [ ]:
# Test MBRS Sandbox — เปรียบเทียบ MF vs LightGCN vs MB-CGCN V1 vs MB-CGCN V2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epochs_v1  = list(range(20, 601, 20))   # eval every 20
epochs_v2  = list(range(50, 601, 50))   # eval every 50

colors = {
    'V1':  '#1f77b4',
    'MF':  '#ff7f0e',
    'GCN': '#2ca02c',
    'V2':  '#d62728',
}

# ── Plot 1: Training Loss (V1, MF, GCN — same dataset) ───────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs_v1, v1_log['loss'],  color=colors['V1'],  lw=2, label='MB-CGCN V1')
ax.plot(epochs_v1, mf_log['loss'],  color=colors['MF'],  lw=2, ls='--', label='Baseline MF')
ax.plot(epochs_v1, gcn_log['loss'], color=colors['GCN'], lw=2, ls='-.', label='Baseline LightGCN')
ax.set_title('Training Loss (V1 dataset)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('BPR Loss')
ax.legend(); ax.grid(ls='--', alpha=0.6)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/0_DATA_RM/plot_loss.png', dpi=150)
plt.show()

# ── Plot 2: Recall@10 / NDCG@10 over epochs (V1 dataset) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Learning Curves — V1 dataset', fontweight='bold')
for ax, key, title in zip(axes, ['recall10', 'ndcg10'], ['Recall@10', 'NDCG@10']):
    ax.plot(epochs_v1, v1_log[key],  color=colors['V1'],  lw=2, label='MB-CGCN V1')
    ax.plot(epochs_v1, mf_log[key],  color=colors['MF'],  lw=2, ls='--', label='Baseline MF')
    ax.plot(epochs_v1, gcn_log[key], color=colors['GCN'], lw=2, ls='-.', label='Baseline LightGCN')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(ls='--', alpha=0.6)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/0_DATA_RM/plot_learning_curves_v1.png', dpi=150)
plt.show()

# ── Plot 3: V2 learning curve (its own dataset) ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('MB-CGCN V2 Learning Curves', fontweight='bold')
for ax, key, title in zip(axes, ['recall10', 'ndcg10'], ['Recall@10', 'NDCG@10']):
    ax.plot(epochs_v2, v2_log[key], color=colors['V2'], lw=2, label='MB-CGCN V2')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(ls='--', alpha=0.6)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/0_DATA_RM/plot_learning_curves_v2.png', dpi=150)
plt.show()

# ── Plot 4: Final bar chart — all 4 models ────────────────────────────────────
model_names = ['Baseline MF', 'Baseline LightGCN', 'MB-CGCN V1', 'MB-CGCN V2']
bar_colors  = [colors['MF'], colors['GCN'], colors['V1'], colors['V2']]
metric_keys = ['Recall@10', 'NDCG@10', 'Recall@20', 'NDCG@20']

vals = {k: [final_results[m][k] for m in model_names] for k in metric_keys}
x     = np.arange(len(model_names))
width = 0.18

fig, ax = plt.subplots(figsize=(12, 6))
for i, mk in enumerate(metric_keys):
    bars = ax.bar(x + (i - 1.5) * width, vals[mk], width, label=mk, alpha=0.85)
    for bar in bars:
        ax.annotate(f'{bar.get_height():.4f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=7)

ax.set_xticks(x); ax.set_xticklabels(model_names)
ax.set_title('Final Performance Comparison — All Models', fontweight='bold')
ax.set_ylabel('Score'); ax.legend(); ax.grid(axis='y', ls='--', alpha=0.6)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/0_DATA_RM/plot_final_comparison.png', dpi=150)
plt.show()
print('All plots saved.')